# 03.1 — GenRec QLoRA Fine-Tuning

**RecSys 2026 Tutorial**: Choosing Between Explainable, Retrieval-Augmented, and LLM-Native Recommenders in Text-Rich Domains

This notebook fine-tunes **Qwen2.5-3B-Instruct** with QLoRA on the Amazon Books
sequential recommendation data, then runs inference on the test set and evaluates.

**Why fine-tuning matters**: The zero-shot generative approach (notebook 03) achieves
HR@10 ≈ 0.5% because the LLM has no knowledge of the item catalog. Fine-tuning on
`(history → next item)` pairs teaches the model item transition patterns, dramatically
improving recommendation quality.

**Requirements**: Colab A100 GPU (40GB VRAM). Runtime → Change runtime type → A100.

**Outputs**: Fine-tuned adapter weights + evaluated predictions, saved to Google Drive
for use in notebook 03.

## 0. Setup

In [1]:
!pip install -q transformers accelerate bitsandbytes peft datasets torch sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.0 MB/s eta 0:00:00:00:0100:01


In [2]:
# ──────────────────────────────────────────────────────
# TEST MODE: set to True to run a quick validation pass
# with a small data subset before committing to the full run.
# Set to False for the real training + full evaluation.
# ──────────────────────────────────────────────────────
TEST_MODE = False

if TEST_MODE:
    N_TRAIN = 50        # training examples
    N_TEST = 20         # test examples for inference
    N_EPOCHS = 1        # training epochs
    BATCH_SIZE = 2
    GRAD_ACCUM = 1
    LOG_EVERY = 5
    print("*** TEST MODE: small subset, 1 epoch, fast validation ***")
else:
    N_TRAIN = None      # use all training data
    N_TEST = None       # use all test data
    N_EPOCHS = 3
    BATCH_SIZE = 4
    GRAD_ACCUM = 8
    LOG_EVERY = 25
    print("*** FULL MODE: all data, 3 epochs ***")

*** FULL MODE: all data, 3 epochs ***


In [3]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU detected. Go to Runtime → Change runtime type → A100.")

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Project root on Google Drive
DRIVE_ROOT = '/content/drive/MyDrive/Foundations of Large Language Models/Final Project'
DATA_DIR = f'{DRIVE_ROOT}/data'
RESULTS_DIR = f'{DRIVE_ROOT}/results'
CACHE_DIR = f'{DRIVE_ROOT}/cache'
OUTPUT_DIR = f'{DRIVE_ROOT}/results/genrec_qlora'

for d in [RESULTS_DIR, CACHE_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

# Verify data files exist
needed = ['genrec_train.json', 'genrec_test.json', 'shared_data.pkl']
for f in needed:
    path = f'{DATA_DIR}/{f}'
    assert os.path.exists(path), f'Missing: {path}'
    print(f'  {f}: {os.path.getsize(path)/1e6:.1f} MB')
print(f'\nAll data files found in {DATA_DIR}')

Mounted at /content/drive
  genrec_train.json: 2.7 MB
  genrec_test.json: 3.0 MB
  shared_data.pkl: 3.0 MB

All data files found in /content/drive/MyDrive/Foundations of Large Language Models/Final Project/data


## 1. Load data

Data files are read directly from the project directory on Google Drive.

In [5]:
import json
import pickle
import numpy as np

with open(f'{DATA_DIR}/genrec_train.json') as f:
    genrec_train = json.load(f)
with open(f'{DATA_DIR}/genrec_test.json') as f:
    genrec_test = json.load(f)
with open(f'{DATA_DIR}/shared_data.pkl', 'rb') as f:
    shared = pickle.load(f)

item_titles = shared['item_titles']
test_ground_truth = shared['test_ground_truth']
user_histories = shared['user_histories']
candidate_pools = shared['candidate_pools']

print(f'Train examples: {len(genrec_train)}')
print(f'Test examples:  {len(genrec_test)}')
print(f'Catalog items:  {len(item_titles)}')
print(f'Test users:     {len(test_ground_truth)}')
print(f'\nSample train example:')
print(json.dumps(genrec_train[0], indent=2))

Train examples: 7288
Test examples:  7288
Catalog items:  6117
Test users:     7288

Sample train example:
{
  "user_idx": 2,
  "instruction": "What book would complement this reading history",
  "input": "Die Trying (Jack Reacher Book 2), Gone Girl: A Novel",
  "output": "Prince Lestat: The Vampire Chronicles"
}


## 2. Load Qwen2.5-3B-Instruct with 4-bit quantization

On A100, the base model loads in 4-bit (~2 GB), leaving ample room for
LoRA weights, optimizer states, and activations during training.

In [6]:
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, TrainingArguments,
    Trainer, BitsAndBytesConfig, DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)
model = prepare_model_for_kbit_training(model)

print(f'Model loaded. VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded. VRAM: 2.69 GB


In [7]:
from peft import PeftModel

ADAPTER_DIR = f'{OUTPUT_DIR}/final_adapter'

if os.path.exists(f'{ADAPTER_DIR}/adapter_config.json'):
    # Load previously trained adapter — skip training
    print(f'Found saved adapter at {ADAPTER_DIR}, loading...')
    model = PeftModel.from_pretrained(model, ADAPTER_DIR)
    SKIP_TRAINING = True
    # Still need lora_config for metadata in the save cell
    lora_config = model.peft_config['default']
    print(f'Adapter loaded. Trainable params restored.')
else:
    # Fresh LoRA — will train
    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        target_modules=['q_proj', 'v_proj'],
        task_type='CAUSAL_LM',
    )
    model = get_peft_model(model, lora_config)
    SKIP_TRAINING = False
    print('No saved adapter found. Will train from scratch.')

model.print_trainable_parameters()

Found saved adapter at /content/drive/MyDrive/Foundations of Large Language Models/Final Project/results/genrec_qlora/final_adapter, loading...
Adapter loaded. Trainable params restored.
trainable params: 0 || all params: 3,087,781,888 || trainable%: 0.0000


<cell_type>markdown</cell_type>## 3. Prepare training data

Format each example as: `{instruction}\n\n### input:\n{history}\n\n### Response:\n{target_title}`

Labels are masked for the prompt portion (set to -100) so the model only learns to predict the target title. This follows the same LoRA fine-tuning pattern from Section 9, adapted for causal language modeling.

In [8]:
if SKIP_TRAINING:
    print('Adapter already loaded — skipping data preparation and training.')
    print('Proceeding directly to inference.')
else:
    from torch.utils.data import DataLoader, Dataset as TorchDataset

    CUTOFF_LEN = 512

    def tokenize_example(example):
        """Tokenize a GenRec example with prompt masking (labels = -100 for prompt tokens)."""
        prompt = f"{example['instruction']}\n\n### input:\n{example['input']}\n\n### Response:\n"
        full = prompt + example['output'] + tokenizer.eos_token
        tokenized = tokenizer(
            full, truncation=True, max_length=CUTOFF_LEN, padding='max_length'
        )
        prompt_ids = tokenizer(
            prompt, truncation=True, max_length=CUTOFF_LEN
        )['input_ids']
        prompt_len = len(prompt_ids)
        labels = [-100] * prompt_len + tokenized['input_ids'][prompt_len:]
        tokenized['labels'] = labels
        return tokenized

    class GenRecDataset(TorchDataset):
        """PyTorch Dataset wrapping tokenized GenRec examples."""
        def __init__(self, examples):
            self.data = [tokenize_example(ex) for ex in examples]
        def __len__(self):
            return len(self.data)
        def __getitem__(self, idx):
            item = self.data[idx]
            return {
                'input_ids': torch.tensor(item['input_ids'], dtype=torch.long),
                'attention_mask': torch.tensor(item['attention_mask'], dtype=torch.long),
                'labels': torch.tensor(item['labels'], dtype=torch.long),
            }

    train_examples = genrec_train[:N_TRAIN] if N_TRAIN is not None else genrec_train
    val_examples = genrec_test[:N_TEST] if N_TEST is not None else genrec_test[:500]

    train_dataset = GenRecDataset(train_examples)
    val_dataset = GenRecDataset(val_examples)

    print(f'Train examples: {len(train_dataset)}')
    print(f'Val examples:   {len(val_dataset)}')

    sample = train_dataset[0]
    n_masked = (sample['labels'] == -100).sum().item()
    n_active = (sample['labels'] != -100).sum().item()
    print(f'Sample: {n_masked} masked (prompt), {n_active} active (response) tokens')

Adapter already loaded — skipping data preparation and training.
Proceeding directly to inference.


<cell_type>markdown</cell_type>## 4. Train (manual loop, following Section 9 pattern)

Manual training loop with AdamW, per-epoch validation loss, and tqdm progress bars.
Same pattern as Section 9 LoRA fine-tuning, adapted for causal language modeling.

In [9]:
if SKIP_TRAINING:
    print('Adapter already loaded — skipping training.')
else:
    from torch.utils.data import DataLoader
    from torch.optim import AdamW
    from tqdm.auto import tqdm

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    optimizer = AdamW(model.parameters(), lr=3e-4)

    for epoch in range(N_EPOCHS):
        # ── TRAIN ──
        model.train()
        total_train_loss = 0.0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{N_EPOCHS} - train"):
            optimizer.zero_grad()

            input_ids = batch['input_ids'].to(model.device)
            attention_mask = batch['attention_mask'].to(model.device)
            labels = batch['labels'].to(model.device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )
            loss = outputs.loss
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / len(train_loader)

        # ── VALIDATION ──
        model.eval()
        total_val_loss = 0.0

        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(model.device)
                attention_mask = batch['attention_mask'].to(model.device)
                labels = batch['labels'].to(model.device)

                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels,
                )
                total_val_loss += outputs.loss.item()

        avg_val_loss = total_val_loss / len(val_loader)

        print(
            f"Epoch {epoch+1}/{N_EPOCHS} "
            f"- train loss: {avg_train_loss:.4f} "
            f"- val loss: {avg_val_loss:.4f}"
        )

Adapter already loaded — skipping training.


In [10]:
if SKIP_TRAINING:
    print(f'Adapter already saved at {ADAPTER_DIR}')
else:
    # Save final adapter weights
    model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
    print(f'Adapter saved to {ADAPTER_DIR}')

    adapter_size = sum(
        os.path.getsize(os.path.join(ADAPTER_DIR, f))
        for f in os.listdir(ADAPTER_DIR)
    ) / 1e6
    print(f'Adapter size: {adapter_size:.1f} MB')

Adapter already saved at /content/drive/MyDrive/Foundations of Large Language Models/Final Project/results/genrec_qlora/final_adapter


## 5. Inference on test set

Generate recommendations for all test users using the fine-tuned model.
Each generated title is matched against the item catalog via fuzzy matching.

In [11]:
import re
import time
from difflib import get_close_matches

# Build title matching index
title_to_idx = {}
for idx, title in item_titles.items():
    title_to_idx[title.lower().strip()] = idx
all_catalog_titles = list(title_to_idx.keys())


def match_title(generated_title, threshold=0.8):
    gen_lower = generated_title.lower().strip()
    if gen_lower in title_to_idx:
        return title_to_idx[gen_lower]
    matches = get_close_matches(gen_lower, all_catalog_titles, n=1, cutoff=threshold)
    return title_to_idx[matches[0]] if matches else -1


def generate_recommendation_finetuned(example):
    """Generate a single title recommendation from a fine-tuned model."""
    prompt = f"{example['instruction']}\n\n### input:\n{example['input']}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    input_len = inputs['input_ids'].shape[1]
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=30,       # titles are short (avg 7.5 words)
            do_sample=False,         # greedy: faster, more deterministic
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    # Take only the first line — fine-tuned GenRec should output a single title
    first_line = text.split('\n')[0].strip()
    # Strip numbering if present
    cleaned = re.sub(r'^\d+\.\s*', '', first_line).strip().strip('"')
    return [cleaned] if cleaned else []


# Quick sanity check
sample = genrec_test[0]
print(f'Input: {sample["input"][:100]}...')
print(f'Ground truth: {sample["output"]}')
gen_titles = generate_recommendation_finetuned(sample)
print(f'Generated: {gen_titles}')

Input: Die Trying (Jack Reacher Book 2), Gone Girl: A Novel, Prince Lestat: The Vampire Chronicles...
Ground truth: The Library at Mount Char: A Novel


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Generated: ['The Nightingale: A Novel']


In [12]:
# Run inference on test set with periodic checkpointing.
# Resilient to kernel restarts — loads partial progress if available.

CHECKPOINT_PATH = f'{RESULTS_DIR}/genrec_finetuned_inference_checkpoint.pkl'

# Load checkpoint if it exists (e.g., after kernel restart)
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH, 'rb') as f:
        ckpt = pickle.load(f)
    predictions = ckpt['predictions']
    generated_titles_dict = ckpt['generated_titles_dict']
    latencies = ckpt['latencies']
    print(f'Resumed from checkpoint: {len(predictions)} examples already done.')
else:
    predictions = {}
    generated_titles_dict = {}
    latencies = []

# Determine which examples still need processing
test_examples = genrec_test  # full test set
remaining = [ex for ex in test_examples if ex['user_idx'] not in predictions]
print(f'Total: {len(test_examples)}, Remaining: {len(remaining)}')

model.eval()
for i, example in enumerate(remaining):
    user_idx = example['user_idx']

    t0 = time.time()
    gen_titles = generate_recommendation_finetuned(example)
    latencies.append(time.time() - t0)

    matched = [match_title(t, threshold=0.7) for t in gen_titles]
    valid = [it for it in matched if it >= 0]
    predictions[user_idx] = valid
    generated_titles_dict[user_idx] = gen_titles

    # Checkpoint every 500 examples
    if (i + 1) % 500 == 0:
        n_hit = sum(1 for v in predictions.values() if v)
        total_done = len(predictions)
        print(f'  {total_done}/{len(test_examples)} | match rate: {n_hit/total_done:.2%} | '
              f'latency: {np.mean(latencies[-500:])*1000:.0f} ms/example')
        with open(CHECKPOINT_PATH, 'wb') as f:
            pickle.dump({
                'predictions': predictions,
                'generated_titles_dict': generated_titles_dict,
                'latencies': latencies,
            }, f)

# Final save
with open(CHECKPOINT_PATH, 'wb') as f:
    pickle.dump({
        'predictions': predictions,
        'generated_titles_dict': generated_titles_dict,
        'latencies': latencies,
    }, f)

n_with_predictions = sum(1 for v in predictions.values() if v)
print(f'\nDone. Match rate: {n_with_predictions}/{len(predictions)} ({n_with_predictions/len(predictions):.2%})')
print(f'Mean latency: {np.mean(latencies)*1000:.0f} ms/example')

Total: 7288, Remaining: 7288
  500/7288 | match rate: 92.60% | latency: 1261 ms/example
  1000/7288 | match rate: 92.90% | latency: 1249 ms/example
  1500/7288 | match rate: 93.00% | latency: 1240 ms/example
  2000/7288 | match rate: 93.35% | latency: 1277 ms/example
  2500/7288 | match rate: 93.56% | latency: 1256 ms/example
  3000/7288 | match rate: 93.70% | latency: 1255 ms/example
  3500/7288 | match rate: 93.63% | latency: 1224 ms/example
  4000/7288 | match rate: 93.42% | latency: 1292 ms/example
  4500/7288 | match rate: 93.38% | latency: 1269 ms/example
  5000/7288 | match rate: 93.42% | latency: 1240 ms/example
  5500/7288 | match rate: 93.45% | latency: 1290 ms/example
  6000/7288 | match rate: 93.40% | latency: 1275 ms/example
  6500/7288 | match rate: 93.35% | latency: 1228 ms/example
  7000/7288 | match rate: 93.33% | latency: 1274 ms/example

Done. Match rate: 6807/7288 (93.40%)
Mean latency: 1257 ms/example


## 6. Evaluate

In [13]:
import math

def hit_at_k(ranked_list, ground_truth, k=10):
    return 1.0 if ground_truth in ranked_list[:k] else 0.0

def ndcg_at_k(ranked_list, ground_truth, k=10):
    for i, item in enumerate(ranked_list[:k]):
        if item == ground_truth:
            return 1.0 / math.log2(i + 2)
    return 0.0

def evaluate_ranking(preds, gt, k_values=None):
    if k_values is None:
        k_values = [5, 10, 20]
    results = {}
    for k in k_values:
        hits, ndcgs = [], []
        for uid, true_item in gt.items():
            if uid not in preds:
                continue
            hits.append(hit_at_k(preds[uid], true_item, k))
            ndcgs.append(ndcg_at_k(preds[uid], true_item, k))
        results[f'HR@{k}'] = np.mean(hits) if hits else 0.0
        results[f'NDCG@{k}'] = np.mean(ndcgs) if ndcgs else 0.0
    return results


# Full-catalog evaluation
ranking_results = evaluate_ranking(predictions, test_ground_truth, k_values=[1, 5, 10, 20])
print(f'Full-catalog ranking ({len(predictions)} users evaluated):')
for m, v in ranking_results.items():
    print(f'  {m}: {v:.4f}')

# Shared-pool evaluation
pool_predictions = {}
for example in genrec_test:
    uid = example['user_idx']
    if uid not in candidate_pools or uid not in generated_titles_dict:
        continue
    pool_items = candidate_pools[uid]
    pool_title_to_idx = {
        item_titles[it].lower().strip(): it
        for it in pool_items if it in item_titles
    }
    pool_titles_list = list(pool_title_to_idx.keys())
    gen_titles = generated_titles_dict.get(uid, [])
    matched = []
    for t in gen_titles:
        gl = t.lower().strip()
        if gl in pool_title_to_idx:
            matched.append(pool_title_to_idx[gl])
            continue
        close = get_close_matches(gl, pool_titles_list, n=1, cutoff=0.7)
        if close:
            matched.append(pool_title_to_idx[close[0]])
    pool_predictions[uid] = matched

pool_ranking_results = evaluate_ranking(pool_predictions, test_ground_truth, k_values=[1, 5, 10, 20])
print(f'\nShared-pool ranking ({len(pool_predictions)} users, 100 candidates/user):')
for m, v in pool_ranking_results.items():
    print(f'  {m}: {v:.4f}')

# Compare with zero-shot baseline
print('\n--- Comparison (full catalog) ---')
print(f'{"Metric":<12} {"Fine-tuned":>12} {"Zero-shot":>12}')
zero_shot = {'HR@10': 0.0050, 'NDCG@10': 0.0032}
for m in ['HR@10', 'NDCG@10']:
    ft = ranking_results.get(m, 0)
    zs = zero_shot.get(m, 0)
    print(f'{m:<12} {ft:>12.4f} {zs:>12.4f}')

Full-catalog ranking (7288 users evaluated):
  HR@1: 0.0266
  NDCG@1: 0.0266
  HR@5: 0.0266
  NDCG@5: 0.0266
  HR@10: 0.0266
  NDCG@10: 0.0266
  HR@20: 0.0266
  NDCG@20: 0.0266

Shared-pool ranking (7288 users, 100 candidates/user):
  HR@1: 0.0445
  NDCG@1: 0.0445
  HR@5: 0.0445
  NDCG@5: 0.0445
  HR@10: 0.0445
  NDCG@10: 0.0445
  HR@20: 0.0445
  NDCG@20: 0.0445

--- Comparison (full catalog) ---
Metric         Fine-tuned    Zero-shot
HR@10              0.0266       0.0050
NDCG@10            0.0266       0.0032


## 7. Qualitative examples

In [14]:
# Show some examples of generated vs ground-truth titles
n_show = 10
n_correct = 0
print(f'{"User":>6}  {"Match":>5}  {"Generated":.<50}  Ground Truth')
print('-' * 120)
for ex in genrec_test[:n_show]:
    uid = ex['user_idx']
    gt_title = ex['output']
    gen = generated_titles_dict.get(uid, ['(none)'])
    gen_top = gen[0] if gen else '(none)'
    # Check if ground truth was matched
    gt_idx = test_ground_truth.get(uid)
    hit = gt_idx in predictions.get(uid, [])
    if hit:
        n_correct += 1
    marker = 'HIT' if hit else ''
    print(f'{uid:>6}  {marker:>5}  {gen_top:.<50}  {gt_title}')

print(f'\nHits in sample: {n_correct}/{n_show}')

  User  Match  Generated.........................................  Ground Truth
------------------------------------------------------------------------------------------------------------------------
     2         The Nightingale: A Novel..........................  The Library at Mount Char: A Novel
     4         The 7 Habits of Highly Effective People: Powerful Lessons in Personal Change  Essentialism: The Disciplined Pursuit of Less
     6         The Very Hungry Caterpillar.......................  The Book with No Pictures
     9         The Girl on the Train: A Novel....................  Brain Quest Workbook: Grade 3
    11         The Last Mile: A Novel............................  The Professor (McMurtrie and Drake Legal Thrillers Book 1)
    15         The 7 Habits of Highly Effective People: Powerful Lessons in Personal Change  Vintage Valentines (Press Out Book)
    16         The Silent Patient................................  Lincoln in the Bardo: A Novel
    18         T

## 8. Save results

Save predictions and results to Google Drive so notebook 03 can load them.

In [15]:
results = {
    'paradigm': 'generative_finetuned',
    'model': f'QLoRA fine-tuned {MODEL_ID}',
    'anchor_paper': 'GenRec (Ji et al., ECIR 2024)',
    'ranking': ranking_results,
    'ranking_pooled': pool_ranking_results,
    'explanation': {},
    'system': {
        'latency': {
            'mean_latency_ms': float(np.mean(latencies) * 1000),
            'p50_latency_ms': float(np.median(latencies) * 1000),
            'p95_latency_ms': float(np.percentile(latencies, 95) * 1000),
        },
        'training': {
            'model': MODEL_ID,
            'lora_r': lora_config.r,
            'lora_alpha': lora_config.lora_alpha,
        },
    },
}

with open(f'{RESULTS_DIR}/generative_finetuned_results.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)
with open(f'{RESULTS_DIR}/generative_finetuned_predictions.pkl', 'wb') as f:
    pickle.dump(predictions, f)
with open(f'{RESULTS_DIR}/generative_finetuned_titles.pkl', 'wb') as f:
    pickle.dump(generated_titles_dict, f)

print(f'Results saved to {RESULTS_DIR}/')

Results saved to /content/drive/MyDrive/Foundations of Large Language Models/Final Project/results/


In [16]:
# Download results locally (optional)
from google.colab import files
files.download(f'{RESULTS_DIR}/generative_finetuned_results.json')
files.download(f'{RESULTS_DIR}/generative_finetuned_predictions.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Summary

This notebook fine-tuned Qwen2.5-3B-Instruct with QLoRA on 7,288 sequential
recommendation examples from Amazon Books.

**Key comparison** (full-catalog HR@10):
- Zero-shot (no fine-tuning): ~0.5%
- QLoRA fine-tuned: see results above

The fine-tuned model learns the item catalog's transition patterns, enabling it
to generate titles that actually exist in the catalog and match user preferences.

**Next steps:**
- Copy `generative_finetuned_results.json` to `Notebooks/results/`
- Update notebook 04 to include fine-tuned results in the comparison table
- The adapter weights can be loaded for inference without retraining:
  ```python
  from peft import PeftModel
  model = AutoModelForCausalLM.from_pretrained(MODEL_ID, ...)
  model = PeftModel.from_pretrained(model, 'path/to/final_adapter')
  ```